<a href="https://colab.research.google.com/github/luverev/Hello-world/blob/main/Despliegue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Despliegue Modelo de Clasificacion**

* Cargamos el modelo
* Cargamos los datos futuros
* Preparar los datos futuros: normalizar, dummies
* Aplicamos el modelo para la predicción

In [1]:
#Cargamos librerías principales

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
variables

array(['Petróleo_t', 'Gas Natural_t', 'Carbón_t',
       'Energía Hidroeléctrica_t', 'Energías Renovables_t',
       'Region_Africa', 'Region_Asia Pacific', 'Region_CIS',
       'Region_Cent. America', 'Region_Europe', 'Region_Middle East',
       'Region_North America'], dtype=object)

In [2]:
#Cargamos el modelo
import pickle
filename = 'modelo-class.pkl'
modelo, min_max_scaler,variables = pickle.load(open(filename, 'rb'))

In [3]:
modelo

KNeighborsClassifier(metric='euclidean', n_neighbors=1)

In [4]:
#Cargamos los datos futuros
data = pd.read_excel("datos_despliegue.xlsx", sheet_name="despliegue")
data.head()


,Country,Region,Year,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t
0,Colombia,Cent. America,2025,0.96,0.45,0.10,0.60,0.08
1,France,Europe,2025,2.92,1.55,0.23,0.55,0.72


In [5]:
#Se realiza copia de los datos antes de la preparacion
data_original=data.copy()

In [6]:
#Eliminamos variables irrelevantes para el modelo predictivo
data = data.drop(columns='Country') # Se desea predecir como número
data = data.drop(columns='Year') # Year no representa una característica energética del país sino una referencia temporal
data.head()

,Region,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t
0,Cent. America,0.96,0.45,0.10,0.60,0.08
1,Europe,2.92,1.55,0.23,0.55,0.72


In [7]:
#Se crean dummies, en despliegue drop_first= False
data = pd.get_dummies(data, columns=['Region'], drop_first=False, dtype=int)
data.head()

,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t,Region_Cent. America,Region_Europe
0,0.96,0.45,0.10,0.60,0.08,1,0
1,2.92,1.55,0.23,0.55,0.72,0,1


In [10]:
#Según el error anterior, se adicionan las columnas faltantes
data=data.reindex(columns=variables,fill_value=0)
data.head()

,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t,Region_Africa,Region_Asia Pacific,Region_CIS,Region_Cent. America,Region_Europe,Region_Middle East,Region_North America
0,0.96,0.45,0.10,0.60,0.08,0,0,0,1,0,0,0
1,2.92,1.55,0.23,0.55,0.72,0,0,0,0,1,0,0


In [11]:
#Se normaliza para predecir con Knn, si tu modelo es Tree o RF se debe comentar esta celda
#En los despliegues no se llama fit, solo transform
variables_numericas=['Petróleo_t','Gas Natural_t','Carbón_t','Energía Hidroeléctrica_t','Energías Renovables_t']

data[variables_numericas]= min_max_scaler.transform(data[variables_numericas])
data.head()

,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t,Region_Africa,Region_Asia Pacific,Region_CIS,Region_Cent. America,Region_Europe,Region_Middle East,Region_North America
0,0.026853,0.014211,0.001139,0.048995,0.005983,0,0,0,1,0,0,0
1,0.081678,0.048950,0.002619,0.044912,0.053849,0,0,0,0,1,0,0


**Predicciones**

In [12]:
#Hacemos la predicción con el Tree
Y_pred = modelo.predict(data)
print(Y_pred)

['Bajo_0.0-0.5' 'Medio_0,51-5']


In [13]:
data_original['Prediccion']=Y_pred
data_original.head()

,Country,Region,Year,Petróleo_t,Gas Natural_t,Carbón_t,Energía Hidroeléctrica_t,Energías Renovables_t,Prediccion
0,Colombia,Cent. America,2025,0.96,0.45,0.10,0.60,0.08,Bajo_0.0-0.5
1,France,Europe,2025,2.92,1.55,0.23,0.55,0.72,"Medio_0,51-5"


In [15]:
#Se guardan las predicciones
data_original.to_excel('predicciones.xlsx', index=False)